<a href="https://www.kaggle.com/code/ahmedmshakil/thesiswork-ml-blockchain-reputation-scoring?scriptVersionId=311957907" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path.cwd()

DATA_FILES = {
    "Ethereum_V2_Transactions.csv": "10sCa8iJKZBpLFjoTLYt6jRx5E160znju",
    "Ethereum_V3_Transactions.csv": "1xjcmE-b1MALMmZt3yngawCVXP2qJfl2V",
}

REQUIRED_PACKAGES = {
    "gdown": "gdown",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "torch": "torch",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
}


def is_colab() -> bool:
    return "google.colab" in sys.modules


def is_kaggle() -> bool:
    return Path("/kaggle").exists()


def ensure_packages(package_map: dict[str, str]) -> None:
    missing = [package for module, package in package_map.items() if importlib.util.find_spec(module) is None]
    if missing:
        print(f"Installing missing packages: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


def candidate_roots() -> list[Path]:
    roots = [
        WORKDIR,
        WORKDIR / "data",
        WORKDIR / "datasets",
        Path("/content"),
        Path("/content/drive/MyDrive"),
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]
    ordered = []
    for root in roots:
        if root.exists() and root not in ordered:
            ordered.append(root)
    return ordered


def find_existing_file(file_name: str) -> Path | None:
    for root in candidate_roots():
        direct = root / file_name
        if direct.exists():
            return direct

    for root in candidate_roots():
        try:
            match = next(root.rglob(file_name))
            return match
        except StopIteration:
            continue
        except Exception:
            continue

    return None


def materialize_dataset(source_path: Path, target_path: Path) -> Path:
    if target_path.exists():
        return target_path

    target_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(source_path.resolve(), target_path)
        print(f"Linked {source_path} -> {target_path}")
    except Exception:
        shutil.copy2(source_path, target_path)
        print(f"Copied {source_path} -> {target_path}")
    return target_path


def ensure_gdown():
    if importlib.util.find_spec("gdown") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown
    return gdown


def ensure_data_files() -> None:
    gdown = None

    for file_name, file_id in DATA_FILES.items():
        target_path = WORKDIR / file_name
        if target_path.exists():
            print(f"Using local file: {target_path}")
            continue

        existing = find_existing_file(file_name)
        if existing is not None:
            materialize_dataset(existing, target_path)
            continue

        if gdown is None:
            gdown = ensure_gdown()

        url = f"https://docs.google.com/uc?export=download&id={file_id}"
        print(f"Downloading {file_name} from Google Drive...")
        download_result = gdown.download(url, str(target_path), quiet=False)
        if not download_result or not target_path.exists():
            raise FileNotFoundError(
                f"Could not obtain {file_name}. On Kaggle without internet, attach the CSVs as a dataset."
            )


ensure_packages(REQUIRED_PACKAGES)
ensure_data_files()

print("Environment bootstrap complete")
print(f"Working directory: {WORKDIR}")
print(f"Platform: {'Colab' if is_colab() else 'Kaggle' if is_kaggle() else 'Local'}")
for file_name in DATA_FILES:
    print(f"Ready: {(WORKDIR / file_name).resolve()}")

Downloading...
From (original): https://docs.google.com/uc?export=download&id=10sCa8iJKZBpLFjoTLYt6jRx5E160znju
From (redirected): https://docs.google.com/uc?export=download&id=10sCa8iJKZBpLFjoTLYt6jRx5E160znju&confirm=t&uuid=3f9465d3-2594-4345-8eae-ae630256b800
To: /kaggle/working/Ethereum_V2_Transactions.csv
100%|██████████| 599M/599M [00:12<00:00, 49.1MB/s] 


Downloading...
From (original): https://docs.google.com/uc?export=download&id=1xjcmE-b1MALMmZt3yngawCVXP2qJfl2V
From (redirected): https://docs.google.com/uc?export=download&id=1xjcmE-b1MALMmZt3yngawCVXP2qJfl2V&confirm=t&uuid=0deade7a-50da-411f-819c-66959a297d38
To: /kaggle/working/Ethereum_V3_Transactions.csv
100%|██████████| 337M/337M [00:10<00:00, 32.4MB/s] 

Environment bootstrap complete
Working directory: /kaggle/working
Platform: Kaggle
Ready: /kaggle/working/Ethereum_V2_Transactions.csv
Ready: /kaggle/working/Ethereum_V3_Transactions.csv


In [2]:
%%writefile 01_data_cleaning_and_validation.py
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd


# =========================================================
# CONFIG
# =========================================================
V2_PATH = "Ethereum_V2_Transactions.csv"
V3_PATH = "Ethereum_V3_Transactions.csv"

OUTPUT_DIR = Path("data/cleaned")
REPORT_DIR = Path("reports")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# HELPERS
# =========================================================
def print_sep():
    print("\n" + "=" * 90 + "\n")


def safe_to_numeric(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def normalize_hex(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.strip()
        .str.lower()
        .replace({"nan": pd.NA, "none": pd.NA, "": pd.NA})
    )


def clean_function_name(s: pd.Series) -> pd.Series:
    s = s.astype("string").fillna("unknown")
    s = s.str.strip()
    s = s.replace({"": "unknown", "<NA>": "unknown", "nan": "unknown"})
    return s


def summarize_missing(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100.0),
        "dtype": df.dtypes.astype(str).values,
    })
    return out.sort_values(["missing_pct", "missing_count"], ascending=False).reset_index(drop=True)


def data_quality_checks(df: pd.DataFrame, dataset_name: str) -> dict:
    checks = {
        "dataset_name": dataset_name,
        "n_rows": int(len(df)),
        "n_cols": int(df.shape[1]),
        "n_duplicate_rows": int(df.duplicated().sum()),
        "n_duplicate_txhash": int(df["TxHash"].duplicated().sum()) if "TxHash" in df.columns else None,
        "n_missing_from": int(df["From"].isna().sum()) if "From" in df.columns else None,
        "n_missing_to": int(df["To"].isna().sum()) if "To" in df.columns else None,
        "n_missing_timestamp": int(df["Timestamp"].isna().sum()) if "Timestamp" in df.columns else None,
        "n_missing_timeStamp": int(df["timeStamp"].isna().sum()) if "timeStamp" in df.columns else None,
        "n_negative_value": int((df["Value"] < 0).sum()) if "Value" in df.columns else None,
        "n_negative_gas": int((df["gas"] < 0).sum()) if "gas" in df.columns else None,
        "n_negative_gasprice": int((df["GasPrice"] < 0).sum()) if "GasPrice" in df.columns else None,
        "n_negative_gasused": int((df["GasUsed"] < 0).sum()) if "GasUsed" in df.columns else None,
        "n_negative_confirmations": int((df["Confirmations"] < 0).sum()) if "Confirmations" in df.columns else None,
        "n_invalid_iserror": int((~df["isError"].isin([0, 1]) & df["isError"].notna()).sum()) if "isError" in df.columns else None,
        "min_timestamp": str(df["Timestamp"].min()) if "Timestamp" in df.columns else None,
        "max_timestamp": str(df["Timestamp"].max()) if "Timestamp" in df.columns else None,
        "n_unique_wallets_from": int(df["From"].nunique(dropna=True)) if "From" in df.columns else None,
        "n_unique_wallets_to": int(df["To"].nunique(dropna=True)) if "To" in df.columns else None,
        "n_unique_methods": int(df["MethodID"].nunique(dropna=True)) if "MethodID" in df.columns else None,
        "n_unique_functions": int(df["FunctionName"].nunique(dropna=True)) if "FunctionName" in df.columns else None,
    }
    return checks


def load_and_clean(csv_path: str, dataset_label: str) -> tuple[pd.DataFrame, dict]:
    print_sep()
    print(f"LOADING {dataset_label}: {csv_path}")

    df = pd.read_csv(csv_path, low_memory=False)
    original_rows = len(df)

    # Remove accidental unnamed index columns if present
    unnamed_cols = [c for c in df.columns if c.lower().startswith("unnamed:")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    # Standardize column names minimally
    df.columns = [c.strip() for c in df.columns]

    # Add dataset label
    df["Dataset"] = dataset_label

    numeric_cols = [
        "BlockNumber",
        "timeStamp",
        "Nonce",
        "TransactionIndex",
        "Value",
        "gas",
        "GasPrice",
        "CumulativeGasUsed",
        "TxReceiptStatus",
        "GasUsed",
        "Confirmations",
        "isError",
    ]
    df = safe_to_numeric(df, numeric_cols)

    # Normalize address/hash-like columns
    for col in ["BlockHash", "TxHash", "From", "To", "MethodID", "ContractAddress"]:
        if col in df.columns:
            df[col] = normalize_hex(df[col])

    # Text columns
    if "FunctionName" in df.columns:
        df["FunctionName"] = clean_function_name(df["FunctionName"])

    if "InputData" in df.columns:
        df["InputData"] = df["InputData"].astype("string")

    # Build Timestamp
    # Prefer existing Timestamp if valid, otherwise derive from epoch timeStamp
    if "Timestamp" in df.columns:
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce", utc=True)
    else:
        df["Timestamp"] = pd.NaT

    if "timeStamp" in df.columns:
        ts_from_epoch = pd.to_datetime(df["timeStamp"], unit="s", errors="coerce", utc=True)
        df["Timestamp"] = df["Timestamp"].fillna(ts_from_epoch)

    # Derived calendar features
    df["Year"] = df["Timestamp"].dt.year
    df["Month"] = df["Timestamp"].dt.month
    df["Day"] = df["Timestamp"].dt.day
    df["Hour"] = df["Timestamp"].dt.hour
    df["Date"] = df["Timestamp"].dt.date.astype("string")

    # Basic canonical ordering
    sort_cols = [c for c in ["Timestamp", "BlockNumber", "TransactionIndex", "TxHash"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)

    # Remove exact duplicate rows
    exact_dups = int(df.duplicated().sum())
    if exact_dups > 0:
        df = df.drop_duplicates().reset_index(drop=True)

    # If duplicate TxHash exists, keep earliest chronologically
    duplicate_txhash_count = 0
    if "TxHash" in df.columns:
        duplicate_txhash_count = int(df["TxHash"].duplicated().sum())
        if duplicate_txhash_count > 0:
            df = df.drop_duplicates(subset=["TxHash"], keep="first").reset_index(drop=True)

    # Wallet role helper
    if "From" in df.columns:
        df["Wallet"] = df["From"]
    else:
        df["Wallet"] = pd.NA

    # Coerce impossible negatives to NaN for fields that should not be negative
    non_negative_cols = [
        "BlockNumber", "Nonce", "TransactionIndex", "Value", "gas", "GasPrice",
        "CumulativeGasUsed", "GasUsed", "Confirmations"
    ]
    for col in non_negative_cols:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    # Standardize binary flags
    if "isError" in df.columns:
        df["isError"] = df["isError"].where(df["isError"].isin([0, 1]), np.nan)

    if "TxReceiptStatus" in df.columns:
        df["TxReceiptStatus"] = df["TxReceiptStatus"].where(df["TxReceiptStatus"].isin([0, 1]), np.nan)

    # Data quality summary
    checks = data_quality_checks(df, dataset_label)
    checks["original_rows"] = int(original_rows)
    checks["exact_duplicates_removed"] = exact_dups
    checks["duplicate_txhash_removed"] = duplicate_txhash_count
    checks["final_rows"] = int(len(df))

    return df, checks


# =========================================================
# MAIN
# =========================================================
def main():
    print_sep()
    print("AAVE DATA CLEANING AND VALIDATION")

    v2_df, v2_checks = load_and_clean(V2_PATH, "V2")
    v3_df, v3_checks = load_and_clean(V3_PATH, "V3")

    print_sep()
    print("MERGING V2 + V3")
    combined_df = pd.concat([v2_df, v3_df], ignore_index=True)

    sort_cols = [c for c in ["Timestamp", "BlockNumber", "TransactionIndex", "TxHash"] if c in combined_df.columns]
    combined_df = combined_df.sort_values(sort_cols).reset_index(drop=True)

    combined_checks = data_quality_checks(combined_df, "COMBINED")
    combined_checks["original_rows"] = int(len(v2_df) + len(v3_df))
    combined_checks["exact_duplicates_removed"] = 0
    combined_checks["duplicate_txhash_removed"] = 0
    combined_checks["final_rows"] = int(len(combined_df))

    # Save cleaned data
    v2_out = OUTPUT_DIR / "aave_v2_cleaned.csv"
    v3_out = OUTPUT_DIR / "aave_v3_cleaned.csv"
    combined_out = OUTPUT_DIR / "aave_combined_cleaned.csv"

    v2_df.to_csv(v2_out, index=False)
    v3_df.to_csv(v3_out, index=False)
    combined_df.to_csv(combined_out, index=False)

    # Save missingness reports
    summarize_missing(v2_df).to_csv(REPORT_DIR / "v2_missingness.csv", index=False)
    summarize_missing(v3_df).to_csv(REPORT_DIR / "v3_missingness.csv", index=False)
    summarize_missing(combined_df).to_csv(REPORT_DIR / "combined_missingness.csv", index=False)

    # Save overall report
    final_report = {
        "v2": v2_checks,
        "v3": v3_checks,
        "combined": combined_checks,
        "columns_in_combined": combined_df.columns.tolist(),
    }

    with open(REPORT_DIR / "data_cleaning_report.json", "w", encoding="utf-8") as f:
        json.dump(final_report, f, indent=2)

    print_sep()
    print("CLEANING COMPLETE")
    print(f"Saved: {v2_out}")
    print(f"Saved: {v3_out}")
    print(f"Saved: {combined_out}")
    print(f"Saved: {REPORT_DIR / 'data_cleaning_report.json'}")
    print(f"Combined rows: {len(combined_df):,}")
    print(f"Combined columns: {combined_df.shape[1]}")

    print_sep()
    print("TOP FUNCTION NAMES")
    if "FunctionName" in combined_df.columns:
        print(combined_df["FunctionName"].value_counts(dropna=False).head(20))

    print_sep()
    print("DONE")


if __name__ == "__main__":
    main()

Writing 01_data_cleaning_and_validation.py


In [3]:
%%writefile 02_eda_and_leakage_audit.py
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# =========================================================
# CONFIG
# =========================================================
INPUT_PATH = "data/cleaned/aave_combined_cleaned.csv"
EDA_DIR = Path("reports/eda")
EDA_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# HELPERS
# =========================================================
def print_sep():
    print("\n" + "=" * 90 + "\n")


def save_plot(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def safe_log1p(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    s = s.where(s >= 0)
    return np.log1p(s)


def compute_numeric_summary(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    existing = [c for c in cols if c in df.columns]
    if not existing:
        return pd.DataFrame()

    summary = df[existing].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    summary["missing_count"] = df[existing].isna().sum()
    summary["missing_pct"] = df[existing].isna().mean() * 100
    return summary.reset_index().rename(columns={"index": "feature"})


def target_feasibility_check(df: pd.DataFrame) -> dict:
    expected_for_hf = [
        "HealthFactor",
        "healthFactor",
        "CurrentATokenBalance",
        "CurrentStableDebt",
        "CurrentVariableDebt",
        "PrincipalStableDebt",
        "ReserveLiquidationThreshold",
        "Ltv",
        "LiquidationThreshold",
    ]

    existing = [c for c in expected_for_hf if c in df.columns]

    message = (
        "HF target is NOT directly constructable from this dataset alone."
        if len(existing) == 0 else
        "Some health-factor-related fields exist, but derivation still needs verification."
    )

    return {
        "has_direct_health_factor_column": any(c in df.columns for c in ["HealthFactor", "healthFactor"]),
        "related_columns_found": existing,
        "can_support_hf_claim_now": False if len(existing) == 0 else False,
        "message": message,
    }


def leakage_audit(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    candidate_cols = [
        "BlockNumber", "Nonce", "TransactionIndex", "Value", "gas", "GasPrice",
        "CumulativeGasUsed", "TxReceiptStatus", "GasUsed", "Confirmations", "isError",
        "Hour", "Day", "Month", "Year"
    ]
    existing = [c for c in candidate_cols if c in df.columns and c != target_col]

    out = []
    for col in existing:
        x = pd.to_numeric(df[col], errors="coerce")
        y = pd.to_numeric(df[target_col], errors="coerce")

        valid = x.notna() & y.notna()
        if valid.sum() < 10:
            corr = np.nan
        else:
            corr = x[valid].corr(y[valid])

        rule_flag = (
            col in {"CumulativeGasUsed", "TxReceiptStatus", "GasUsed", "TransactionIndex", "BlockNumber"}
        )

        out.append({
            "feature": col,
            "abs_pearson_corr_with_target": abs(corr) if pd.notna(corr) else np.nan,
            "signed_pearson_corr_with_target": corr,
            "hard_rule_leakage_flag": rule_flag,
            "reason": (
                "post-execution or block-placement signal"
                if rule_flag else
                "needs contextual review"
            )
        })

    out_df = pd.DataFrame(out)
    if not out_df.empty:
        out_df = out_df.sort_values(
            ["hard_rule_leakage_flag", "abs_pearson_corr_with_target"],
            ascending=[False, False]
        ).reset_index(drop=True)
    return out_df


def plot_top_functions(df: pd.DataFrame, out_path: Path, top_n: int = 15):
    if "FunctionName" not in df.columns:
        return
    vc = df["FunctionName"].fillna("unknown").value_counts().head(top_n)
    plt.figure(figsize=(12, 6))
    vc.sort_values().plot(kind="barh")
    plt.xlabel("Count")
    plt.ylabel("FunctionName")
    plt.title("Top Function Names")
    save_plot(out_path)


def plot_dataset_split(df: pd.DataFrame, out_path: Path):
    if "Dataset" not in df.columns:
        return
    vc = df["Dataset"].value_counts(dropna=False)
    plt.figure(figsize=(6, 6))
    plt.pie(vc.values, labels=vc.index.astype(str), autopct="%1.1f%%")
    plt.title("Dataset Composition")
    save_plot(out_path)


def plot_transactions_over_time(df: pd.DataFrame, out_path: Path):
    if "Timestamp" not in df.columns:
        return
    tmp = df.copy()
    tmp["Timestamp"] = pd.to_datetime(tmp["Timestamp"], errors="coerce", utc=True)
    tmp = tmp.dropna(subset=["Timestamp"])
    if tmp.empty:
        return
    by_date = tmp.groupby(tmp["Timestamp"].dt.date).size()
    plt.figure(figsize=(14, 6))
    plt.plot(by_date.index, by_date.values)
    plt.xlabel("Date")
    plt.ylabel("Transaction Count")
    plt.title("Transactions Over Time")
    plt.xticks(rotation=45)
    save_plot(out_path)


def plot_wallet_activity(df: pd.DataFrame, out_path: Path):
    if "From" not in df.columns:
        return
    vc = df["From"].value_counts().head(1000)
    log_counts = np.log1p(vc.values)
    plt.figure(figsize=(10, 6))
    plt.hist(log_counts, bins=50)
    plt.xlabel("log(1 + transactions per wallet)")
    plt.ylabel("Frequency")
    plt.title("Wallet Activity Distribution")
    save_plot(out_path)


def plot_feature_distribution(df: pd.DataFrame, col: str, out_path: Path, log_transform: bool = False):
    if col not in df.columns:
        return
    vals = pd.to_numeric(df[col], errors="coerce")
    vals = vals.dropna()
    if vals.empty:
        return

    if log_transform:
        vals = np.log1p(vals[vals >= 0])

    plt.figure(figsize=(10, 6))
    plt.hist(vals, bins=60)
    plt.xlabel(f"{'log1p(' + col + ')' if log_transform else col}")
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {col}")
    save_plot(out_path)


def plot_hourly_pattern(df: pd.DataFrame, out_path: Path):
    if "Hour" not in df.columns:
        return
    vc = df["Hour"].value_counts().sort_index()
    plt.figure(figsize=(10, 5))
    plt.bar(vc.index.astype(int), vc.values)
    plt.xlabel("Hour of Day")
    plt.ylabel("Transaction Count")
    plt.title("Hourly Transaction Pattern")
    save_plot(out_path)


def plot_missingness(df: pd.DataFrame, out_path: Path, top_n: int = 20):
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False).head(top_n)
    plt.figure(figsize=(12, 6))
    missing_pct.sort_values().plot(kind="barh")
    plt.xlabel("Missing %")
    plt.ylabel("Column")
    plt.title(f"Top {top_n} Columns by Missingness")
    save_plot(out_path)


def plot_target_comparison_by_dataset(df: pd.DataFrame, target_col: str, out_path: Path):
    if "Dataset" not in df.columns or target_col not in df.columns:
        return

    vals = []
    labels = []
    for ds in sorted(df["Dataset"].dropna().unique()):
        s = pd.to_numeric(df.loc[df["Dataset"] == ds, target_col], errors="coerce").dropna()
        if not s.empty:
            vals.append(np.log1p(s[s >= 0]))
            labels.append(str(ds))

    if not vals:
        return

    plt.figure(figsize=(8, 6))
    plt.boxplot(vals, labels=labels)
    plt.ylabel(f"log1p({target_col})")
    plt.title(f"{target_col} Distribution by Dataset")
    save_plot(out_path)


# =========================================================
# MAIN
# =========================================================
def main():
    print_sep()
    print("EDA AND LEAKAGE AUDIT")

    df = pd.read_csv(INPUT_PATH, low_memory=False)

    if "Timestamp" in df.columns:
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce", utc=True)

    print(f"Loaded rows: {len(df):,}")
    print(f"Loaded cols: {df.shape[1]}")

    # -----------------------------
    # Base summaries
    # -----------------------------
    numeric_cols = [
        "BlockNumber", "timeStamp", "Nonce", "TransactionIndex", "Value", "gas", "GasPrice",
        "CumulativeGasUsed", "TxReceiptStatus", "GasUsed", "Confirmations", "isError",
        "Hour", "Day", "Month", "Year"
    ]
    numeric_summary = compute_numeric_summary(df, numeric_cols)
    numeric_summary.to_csv(EDA_DIR / "numeric_summary.csv", index=False)

    missing_summary = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100.0),
        "dtype": df.dtypes.astype(str).values,
    }).sort_values("missing_pct", ascending=False)
    missing_summary.to_csv(EDA_DIR / "missing_summary.csv", index=False)

    # -----------------------------
    # Leakage audit for current target
    # -----------------------------
    current_target = "GasUsed" if "GasUsed" in df.columns else None
    if current_target is not None:
        leak_df = leakage_audit(df, current_target)
        leak_df.to_csv(EDA_DIR / "leakage_audit_gasused.csv", index=False)
    else:
        leak_df = pd.DataFrame()

    # -----------------------------
    # HF feasibility check
    # -----------------------------
    hf_check = target_feasibility_check(df)
    with open(EDA_DIR / "health_factor_feasibility.json", "w", encoding="utf-8") as f:
        json.dump(hf_check, f, indent=2)

    # -----------------------------
    # Function / wallet summaries
    # -----------------------------
    if "FunctionName" in df.columns:
        df["FunctionName"].fillna("unknown").value_counts().to_csv(EDA_DIR / "function_name_counts.csv", header=["count"])

    if "From" in df.columns:
        wallet_counts = df["From"].value_counts()
        wallet_counts.describe().to_frame(name="value").to_csv(EDA_DIR / "wallet_activity_summary.csv")

    # -----------------------------
    # Plots
    # -----------------------------
    plot_dataset_split(df, EDA_DIR / "dataset_composition.png")
    plot_transactions_over_time(df, EDA_DIR / "transactions_over_time.png")
    plot_wallet_activity(df, EDA_DIR / "wallet_activity_distribution.png")
    plot_top_functions(df, EDA_DIR / "top_functions.png")
    plot_hourly_pattern(df, EDA_DIR / "hourly_pattern.png")
    plot_missingness(df, EDA_DIR / "missingness_top20.png")

    for col in ["Value", "gas", "GasPrice", "GasUsed", "CumulativeGasUsed", "Confirmations", "Nonce"]:
        if col in df.columns:
            plot_feature_distribution(df, col, EDA_DIR / f"{col}_distribution.png", log_transform=False)
            plot_feature_distribution(df, col, EDA_DIR / f"{col}_distribution_log1p.png", log_transform=True)

    if current_target is not None:
        plot_target_comparison_by_dataset(df, current_target, EDA_DIR / "gasused_by_dataset_boxplot.png")

    # -----------------------------
    # Console report
    # -----------------------------
    print_sep()
    print("TOP 15 FUNCTION NAMES")
    if "FunctionName" in df.columns:
        print(df["FunctionName"].fillna("unknown").value_counts().head(15))

    print_sep()
    print("HF FEASIBILITY CHECK")
    print(json.dumps(hf_check, indent=2))

    print_sep()
    print("LIKELY LEAKAGE FEATURES FOR CURRENT TARGET = GasUsed")
    if not leak_df.empty:
        print(leak_df.head(15).to_string(index=False))

    print_sep()
    print("EDA COMPLETE")
    print(f"Saved outputs to: {EDA_DIR}")


if __name__ == "__main__":
    main()

Writing 02_eda_and_leakage_audit.py


In [4]:
!python 01_data_cleaning_and_validation.py
!python 02_eda_and_leakage_audit.py



AAVE DATA CLEANING AND VALIDATION


LOADING V2: Ethereum_V2_Transactions.csv


LOADING V3: Ethereum_V3_Transactions.csv


MERGING V2 + V3


CLEANING COMPLETE
Saved: data/cleaned/aave_v2_cleaned.csv
Saved: data/cleaned/aave_v3_cleaned.csv
Saved: data/cleaned/aave_combined_cleaned.csv
Saved: reports/data_cleaning_report.json
Combined rows: 1,100,245
Combined columns: 28


TOP FUNCTION NAMES
FunctionName
borrow(address asset, uint256 amount, uint256 interestRateMode, uint16 referralCode, address onBehalfOf)                                                       323660
deposit(address asset, uint256 amount, address onBehalfOf, uint16 referralCode)                                                                                230282
withdraw(address token, uint256 amount, address destination)                                                                                                   212233
repay(address _owner, uint256 _pid, uint256 _amount, address _payer)                           

In [5]:
%%writefile 03_train_transaction_behavior_models.py
from __future__ import annotations

import json
import os
import time
from copy import deepcopy
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

XGBOOST_IMPORT_ERROR = None
LIGHTGBM_IMPORT_ERROR = None

try:
    import xgboost as xgb
except Exception as exc:
    xgb = None
    XGBOOST_IMPORT_ERROR = str(exc)

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except Exception as exc:
    lgb = None
    LGBMRegressor = None
    LIGHTGBM_IMPORT_ERROR = str(exc)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


INPUT_PATH = Path(os.environ.get("AAVE_INPUT_PATH", "data/cleaned/aave_combined_cleaned.csv"))
OUTPUT_ROOT = Path(os.environ.get("MODEL_OUTPUT_ROOT", "model_outputs"))
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
RUN_STATE_PATH = CHECKPOINT_DIR / "training_state.json"
PARTIAL_RESULTS_PATH = OUTPUT_ROOT / "all_model_results.partial.csv"
FINAL_RESULTS_PATH = OUTPUT_ROOT / "all_model_results.csv"
RESUME_ENABLED = os.environ.get("RESUME_TRAINING", "1") != "0"
MLP_CHECKPOINT_EVERY = 5
MAX_MLP_EPOCHS = 100

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS = 5
TEMPORAL_TEST_SIZE = 0.20
TARGET = "GasUsed"

CLEAN_FEATURES = [
    "Nonce",
    "Value",
    "gas",
    "GasPrice",
    "Confirmations",
    "isError",
    "Hour",
    "Day",
    "Month",
    "Year",
]

LEAKY_FEATURES = CLEAN_FEATURES + [
    "TransactionIndex",
    "BlockNumber",
    "CumulativeGasUsed",
]

CATEGORICAL_TEXT_FEATURES = [
    "Dataset",
    "MethodID",
    "FunctionName",
]

GROUP_COL = "Wallet"
TIME_COL = "Timestamp"


def detect_hardware() -> dict[str, object]:
    info = {
        "torch_cuda_available": torch.cuda.is_available(),
        "gpu_count": 0,
        "gpu_names": [],
        "use_gpu_for_torch": False,
        "use_gpu_for_xgboost": False,
        "use_gpu_for_lightgbm": False,
    }

    if torch.cuda.is_available():
        info["gpu_count"] = torch.cuda.device_count()
        info["gpu_names"] = [torch.cuda.get_device_name(index) for index in range(info["gpu_count"])]
        info["use_gpu_for_torch"] = True
        info["use_gpu_for_xgboost"] = True
        info["use_gpu_for_lightgbm"] = True

    return info


HARDWARE = detect_hardware()
DEVICE = torch.device("cuda" if HARDWARE["use_gpu_for_torch"] else "cpu")


def print_sep() -> None:
    print("\n" + "=" * 100 + "\n")


def temp_path(path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    return path.with_name(f".tmp-{path.name}")


def atomic_write_text(text: str, path: Path) -> None:
    tmp_path = temp_path(path)
    tmp_path.write_text(text, encoding="utf-8")
    os.replace(tmp_path, path)


def atomic_write_json(payload: dict[str, object], path: Path) -> None:
    atomic_write_text(json.dumps(payload, indent=2, default=str), path)


def atomic_write_csv(frame: pd.DataFrame, path: Path, *, index: bool = False) -> None:
    tmp_path = temp_path(path)
    frame.to_csv(tmp_path, index=index)
    os.replace(tmp_path, path)


def atomic_joblib_dump(payload: object, path: Path) -> None:
    tmp_path = temp_path(path)
    joblib.dump(payload, tmp_path)
    os.replace(tmp_path, path)


def atomic_torch_save(payload: object, path: Path) -> None:
    tmp_path = temp_path(path)
    torch.save(payload, tmp_path)
    os.replace(tmp_path, path)


def atomic_save_current_figure(path: Path) -> None:
    tmp_path = temp_path(path)
    plt.savefig(tmp_path, dpi=300, bbox_inches="tight", format=path.suffix.lstrip("."))
    os.replace(tmp_path, path)


def load_json(path: Path) -> dict[str, object] | None:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def clean_reason(exc: Exception | str) -> str:
    reason = str(exc).strip()
    return reason[:4000] if len(reason) > 4000 else reason


def blank_metrics(status: str = "pending", reason: str | None = None) -> dict[str, object]:
    return {
        "cv_mean_r2": None,
        "cv_std_r2": None,
        "R2": None,
        "RMSE": None,
        "MAE": None,
        "MSE": None,
        "status": status,
        "reason": reason,
    }


def completed_metrics(metrics: dict[str, object]) -> dict[str, object]:
    payload = blank_metrics(status="completed", reason=None)
    payload.update(metrics)
    payload["status"] = payload.get("status") or "completed"
    payload["reason"] = payload.get("reason")
    return payload


def skipped_metrics(reason: str) -> dict[str, object]:
    return blank_metrics(status="skipped", reason=clean_reason(reason))


def failed_metrics(reason: str) -> dict[str, object]:
    return blank_metrics(status="failed", reason=clean_reason(reason))


def metrics_file(output_dir: Path, tag: str) -> Path:
    return output_dir / f"{tag}_test_metrics.json"


def status_file(output_dir: Path, tag: str) -> Path:
    return output_dir / f"{tag}_status.json"


def mlp_checkpoint_file(output_dir: Path, tag: str) -> Path:
    return output_dir / f"{tag}_training_checkpoint.pt"


def save_metrics_bundle(output_dir: Path, tag: str, metrics: dict[str, object]) -> None:
    atomic_write_json(metrics, metrics_file(output_dir, tag))
    atomic_write_json(
        {
            "status": metrics.get("status"),
            "reason": metrics.get("reason"),
            "updated_at": time.time(),
        },
        status_file(output_dir, tag),
    )


def load_saved_metrics(output_dir: Path, tag: str) -> dict[str, object] | None:
    if not RESUME_ENABLED:
        return None

    payload = load_json(metrics_file(output_dir, tag))
    if not isinstance(payload, dict):
        return None

    if payload.get("status") in {"completed", "skipped"}:
        return payload

    return None


def load_run_state() -> dict[str, object]:
    payload = load_json(RUN_STATE_PATH)
    if isinstance(payload, dict):
        payload.setdefault("setups", {})
        return payload
    return {"created_at": time.time(), "setups": {}}


def save_run_state(state: dict[str, object]) -> None:
    state["updated_at"] = time.time()
    atomic_write_json(state, RUN_STATE_PATH)


def set_current_model(state: dict[str, object], setup_name: str, model_name: str) -> None:
    state["current"] = {
        "setup": setup_name,
        "model": model_name,
        "started_at": time.time(),
    }
    save_run_state(state)


def update_model_state(
    state: dict[str, object],
    setup_name: str,
    model_name: str,
    output_dir: Path,
    metrics: dict[str, object],
) -> None:
    setups = state.setdefault("setups", {})
    setup_state = setups.setdefault(setup_name, {"models": {}})
    models = setup_state.setdefault("models", {})
    models[model_name] = {
        "status": metrics.get("status"),
        "reason": metrics.get("reason"),
        "metrics_file": str(metrics_file(output_dir, setup_name)),
        "updated_at": time.time(),
    }
    save_run_state(state)


def save_json(payload: dict[str, object], path: Path) -> None:
    atomic_write_json(payload, path)


def evaluate_regression(y_true, y_pred) -> dict[str, float]:
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        "R2": float(r2),
        "RMSE": float(rmse),
        "MAE": float(mae),
        "MSE": float(mse),
    }


def add_behavior_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    df = df.copy()

    numeric_cols = [
        "Nonce", "Value", "gas", "GasPrice", "Confirmations",
        "isError", "Hour", "Day", "Month", "Year",
        "TransactionIndex", "BlockNumber", "CumulativeGasUsed", "GasUsed",
    ]
    for column in numeric_cols:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    extended_features: list[str] = []

    if "Wallet" in df.columns:
        df["wallet_tx_count_so_far"] = df.groupby("Wallet").cumcount()
        extended_features.append("wallet_tx_count_so_far")

    if "FunctionName" in df.columns:
        df["function_count_so_far"] = df.groupby("FunctionName").cumcount()
        extended_features.append("function_count_so_far")

    if "Value" in df.columns:
        df["log_Value"] = np.log1p(df["Value"].clip(lower=0))
        extended_features.append("log_Value")

    if "gas" in df.columns:
        df["log_gas"] = np.log1p(df["gas"].clip(lower=0))
        extended_features.append("log_gas")

    if "GasPrice" in df.columns:
        df["log_GasPrice"] = np.log1p(df["GasPrice"].clip(lower=0))
        extended_features.append("log_GasPrice")

    clean_features = CLEAN_FEATURES + extended_features
    leaky_features = LEAKY_FEATURES + extended_features
    return df, clean_features, leaky_features


def encode_categoricals(train_df, valid_df, test_df, cat_cols):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    for column in cat_cols:
        if column not in train_df.columns:
            continue

        train_df[column] = train_df[column].astype("string").fillna("missing")
        valid_df[column] = valid_df[column].astype("string").fillna("missing")
        test_df[column] = test_df[column].astype("string").fillna("missing")

        all_train_values = sorted(train_df[column].unique())
        mapping = {value: index for index, value in enumerate(all_train_values)}

        train_df[column] = train_df[column].map(mapping).fillna(-1).astype(int)
        valid_df[column] = valid_df[column].map(mapping).fillna(-1).astype(int)
        test_df[column] = test_df[column].map(mapping).fillna(-1).astype(int)

    return train_df, valid_df, test_df


def prepare_dataset(df: pd.DataFrame, feature_cols: list[str]):
    cols_needed = list(set(feature_cols + [TARGET, GROUP_COL, TIME_COL] + CATEGORICAL_TEXT_FEATURES))
    cols_needed = [column for column in cols_needed if column in df.columns]

    work = df[cols_needed].copy()
    work = work.sort_values(TIME_COL).reset_index(drop=True)

    split_idx = int(len(work) * (1.0 - TEMPORAL_TEST_SIZE))
    train_df = work.iloc[:split_idx].copy()
    test_df = work.iloc[split_idx:].copy()

    train_df = train_df.dropna(subset=[TARGET, GROUP_COL, TIME_COL]).reset_index(drop=True)
    test_df = test_df.dropna(subset=[TARGET, GROUP_COL, TIME_COL]).reset_index(drop=True)

    train_df, _, test_df = encode_categoricals(train_df, train_df.copy(), test_df, CATEGORICAL_TEXT_FEATURES)

    final_features = [column for column in feature_cols if column in train_df.columns]
    final_features += [column for column in CATEGORICAL_TEXT_FEATURES if column in train_df.columns]
    final_features = list(dict.fromkeys(final_features))

    medians = train_df[final_features].median(numeric_only=True)
    train_df[final_features] = train_df[final_features].fillna(medians)
    test_df[final_features] = test_df[final_features].fillna(medians)

    return train_df, test_df, final_features


def grouped_cv_splits(train_df: pd.DataFrame, n_folds: int = 5):
    gkf = GroupKFold(n_splits=n_folds)
    x_dummy = np.zeros(len(train_df))
    groups = train_df[GROUP_COL].values

    splits = []
    for fold, (train_index, valid_index) in enumerate(gkf.split(x_dummy, groups=groups), start=1):
        splits.append((fold, train_index, valid_index))
    return splits


def plot_predictions(y_true, y_pred, out_path: Path, title: str) -> None:
    plt.figure(figsize=(8, 8))
    plt.scatter(y_true, y_pred, alpha=0.3, s=10)
    minimum = min(np.min(y_true), np.min(y_pred))
    maximum = max(np.max(y_true), np.max(y_pred))
    plt.plot([minimum, maximum], [minimum, maximum], "r--", linewidth=2)
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.tight_layout()
    atomic_save_current_figure(out_path)
    plt.close()


def plot_residuals(y_true, y_pred, out_path: Path, title: str) -> None:
    residuals = y_true - y_pred
    plt.figure(figsize=(8, 6))
    plt.scatter(y_pred, residuals, alpha=0.3, s=10)
    plt.axhline(0, linestyle="--", linewidth=2)
    plt.xlabel("Predicted")
    plt.ylabel("Residual")
    plt.title(title)
    plt.tight_layout()
    atomic_save_current_figure(out_path)
    plt.close()


def plot_feature_importance(importance_df: pd.DataFrame | None, out_path: Path, title: str, top_n: int = 15) -> None:
    if importance_df is None or importance_df.empty:
        return
    top = importance_df.sort_values("importance", ascending=False).head(top_n).iloc[::-1]
    plt.figure(figsize=(10, 6))
    plt.barh(top["feature"], top["importance"])
    plt.xlabel("Importance")
    plt.title(title)
    plt.tight_layout()
    atomic_save_current_figure(out_path)
    plt.close()


def save_predictions(y_true, y_pred, output_dir: Path, tag: str) -> None:
    frame = pd.DataFrame(
        {
            "Actual": y_true,
            "Predicted": y_pred,
            "Residual": y_true - y_pred,
        }
    )
    atomic_write_csv(frame, output_dir / f"{tag}_test_predictions.csv", index=False)


def train_random_forest(train_df, test_df, features, output_dir: Path, tag: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    cv_rows = []

    for fold, train_index, valid_index in grouped_cv_splits(train_df, N_FOLDS):
        train_fold = train_df.iloc[train_index]
        valid_fold = train_df.iloc[valid_index]

        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=18,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        model.fit(train_fold[features], train_fold[TARGET])
        pred = model.predict(valid_fold[features])
        metrics = evaluate_regression(valid_fold[TARGET], pred)
        metrics["Fold"] = fold
        cv_rows.append(metrics)

    cv_df = pd.DataFrame(cv_rows)
    atomic_write_csv(cv_df, output_dir / f"{tag}_cv_results.csv", index=False)

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=18,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    model.fit(train_df[features], train_df[TARGET])
    test_pred = model.predict(test_df[features])
    test_metrics = evaluate_regression(test_df[TARGET], test_pred)

    importance_df = pd.DataFrame({"feature": features, "importance": model.feature_importances_}).sort_values(
        "importance", ascending=False
    )

    plot_predictions(test_df[TARGET].values, test_pred, output_dir / f"{tag}_actual_vs_pred.png", f"RF {tag} Actual vs Predicted")
    plot_residuals(test_df[TARGET].values, test_pred, output_dir / f"{tag}_residuals.png", f"RF {tag} Residuals")
    plot_feature_importance(importance_df, output_dir / f"{tag}_feature_importance.png", f"RF {tag} Feature Importance")

    save_predictions(test_df[TARGET].values, test_pred, output_dir, tag)
    atomic_write_csv(importance_df, output_dir / f"{tag}_feature_importance.csv", index=False)
    atomic_joblib_dump(model, output_dir / f"{tag}_model.joblib")

    return completed_metrics(
        {
            "cv_mean_r2": float(cv_df["R2"].mean()),
            "cv_std_r2": float(cv_df["R2"].std()),
            **test_metrics,
        }
    )


def train_adaboost(train_df, test_df, features, output_dir: Path, tag: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    cv_rows = []

    for fold, train_index, valid_index in grouped_cv_splits(train_df, N_FOLDS):
        train_fold = train_df.iloc[train_index]
        valid_fold = train_df.iloc[valid_index]

        scaler = StandardScaler()
        x_train = scaler.fit_transform(train_fold[features])
        x_valid = scaler.transform(valid_fold[features])

        base = DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE)
        model = AdaBoostRegressor(
            estimator=base,
            n_estimators=100,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
        )
        model.fit(x_train, train_fold[TARGET])
        pred = model.predict(x_valid)
        metrics = evaluate_regression(valid_fold[TARGET], pred)
        metrics["Fold"] = fold
        cv_rows.append(metrics)

    cv_df = pd.DataFrame(cv_rows)
    atomic_write_csv(cv_df, output_dir / f"{tag}_cv_results.csv", index=False)

    scaler = StandardScaler()
    x_train = scaler.fit_transform(train_df[features])
    x_test = scaler.transform(test_df[features])

    base = DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE)
    model = AdaBoostRegressor(
        estimator=base,
        n_estimators=150,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
    )
    model.fit(x_train, train_df[TARGET])
    test_pred = model.predict(x_test)
    test_metrics = evaluate_regression(test_df[TARGET], test_pred)

    importance_df = pd.DataFrame({"feature": features, "importance": model.feature_importances_}).sort_values(
        "importance", ascending=False
    )

    plot_predictions(test_df[TARGET].values, test_pred, output_dir / f"{tag}_actual_vs_pred.png", f"AdaBoost {tag} Actual vs Predicted")
    plot_residuals(test_df[TARGET].values, test_pred, output_dir / f"{tag}_residuals.png", f"AdaBoost {tag} Residuals")
    plot_feature_importance(importance_df, output_dir / f"{tag}_feature_importance.png", f"AdaBoost {tag} Feature Importance")

    save_predictions(test_df[TARGET].values, test_pred, output_dir, tag)
    atomic_write_csv(importance_df, output_dir / f"{tag}_feature_importance.csv", index=False)
    atomic_joblib_dump({"model": model, "scaler": scaler}, output_dir / f"{tag}_model.joblib")

    return completed_metrics(
        {
            "cv_mean_r2": float(cv_df["R2"].mean()),
            "cv_std_r2": float(cv_df["R2"].std()),
            **test_metrics,
        }
    )


def train_xgboost(train_df, test_df, features, output_dir: Path, tag: str):
    if xgb is None:
        return skipped_metrics(XGBOOST_IMPORT_ERROR or "XGBoost is unavailable in this runtime")

    output_dir.mkdir(parents=True, exist_ok=True)
    cv_rows = []

    params = {
        "max_depth": 8,
        "eta": 0.05,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "verbosity": 0,
        "seed": RANDOM_STATE,
    }
    if HARDWARE["use_gpu_for_xgboost"]:
        params["device"] = "cuda"

    for fold, train_index, valid_index in grouped_cv_splits(train_df, N_FOLDS):
        train_fold = train_df.iloc[train_index]
        valid_fold = train_df.iloc[valid_index]

        dtrain = xgb.DMatrix(train_fold[features], label=train_fold[TARGET])
        dvalid = xgb.DMatrix(valid_fold[features], label=valid_fold[TARGET])

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=500,
            evals=[(dvalid, "valid")],
            early_stopping_rounds=30,
            verbose_eval=False,
        )

        pred = model.predict(dvalid)
        metrics = evaluate_regression(valid_fold[TARGET], pred)
        metrics["Fold"] = fold
        cv_rows.append(metrics)

    cv_df = pd.DataFrame(cv_rows)
    atomic_write_csv(cv_df, output_dir / f"{tag}_cv_results.csv", index=False)

    dtrain = xgb.DMatrix(train_df[features], label=train_df[TARGET])
    dtest = xgb.DMatrix(test_df[features], label=test_df[TARGET])

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=500,
        evals=[(dtest, "test")],
        early_stopping_rounds=30,
        verbose_eval=False,
    )

    test_pred = model.predict(dtest)
    test_metrics = evaluate_regression(test_df[TARGET], test_pred)

    score = model.get_score(importance_type="gain")
    importance_df = pd.DataFrame({"feature": list(score.keys()), "importance": list(score.values())})
    if not importance_df.empty:
        importance_df = importance_df.sort_values("importance", ascending=False)

    plot_predictions(test_df[TARGET].values, test_pred, output_dir / f"{tag}_actual_vs_pred.png", f"XGBoost {tag} Actual vs Predicted")
    plot_residuals(test_df[TARGET].values, test_pred, output_dir / f"{tag}_residuals.png", f"XGBoost {tag} Residuals")
    plot_feature_importance(importance_df, output_dir / f"{tag}_feature_importance.png", f"XGBoost {tag} Feature Importance")

    save_predictions(test_df[TARGET].values, test_pred, output_dir, tag)
    if not importance_df.empty:
        atomic_write_csv(importance_df, output_dir / f"{tag}_feature_importance.csv", index=False)
    model.save_model(str(output_dir / f"{tag}_model.json"))

    return completed_metrics(
        {
            "cv_mean_r2": float(cv_df["R2"].mean()),
            "cv_std_r2": float(cv_df["R2"].std()),
            **test_metrics,
        }
    )


def train_lightgbm(train_df, test_df, features, output_dir: Path, tag: str):
    if lgb is None or LGBMRegressor is None:
        return skipped_metrics(LIGHTGBM_IMPORT_ERROR or "LightGBM is unavailable in this runtime")

    output_dir.mkdir(parents=True, exist_ok=True)
    cv_rows = []
    use_gpu = HARDWARE["use_gpu_for_lightgbm"]

    for fold, train_index, valid_index in grouped_cv_splits(train_df, N_FOLDS):
        train_fold = train_df.iloc[train_index]
        valid_fold = train_df.iloc[valid_index]

        model = LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            device="gpu" if use_gpu else "cpu",
            verbosity=-1,
        )
        model.fit(
            train_fold[features],
            train_fold[TARGET],
            eval_set=[(valid_fold[features], valid_fold[TARGET])],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)],
        )

        pred = model.predict(valid_fold[features])
        metrics = evaluate_regression(valid_fold[TARGET], pred)
        metrics["Fold"] = fold
        cv_rows.append(metrics)

    cv_df = pd.DataFrame(cv_rows)
    atomic_write_csv(cv_df, output_dir / f"{tag}_cv_results.csv", index=False)

    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        device="gpu" if use_gpu else "cpu",
        verbosity=-1,
    )
    model.fit(
        train_df[features],
        train_df[TARGET],
        eval_set=[(test_df[features], test_df[TARGET])],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)],
    )

    test_pred = model.predict(test_df[features])
    test_metrics = evaluate_regression(test_df[TARGET], test_pred)
    importance_df = pd.DataFrame({"feature": features, "importance": model.feature_importances_}).sort_values(
        "importance", ascending=False
    )

    plot_predictions(test_df[TARGET].values, test_pred, output_dir / f"{tag}_actual_vs_pred.png", f"LightGBM {tag} Actual vs Predicted")
    plot_residuals(test_df[TARGET].values, test_pred, output_dir / f"{tag}_residuals.png", f"LightGBM {tag} Residuals")
    plot_feature_importance(importance_df, output_dir / f"{tag}_feature_importance.png", f"LightGBM {tag} Feature Importance")

    save_predictions(test_df[TARGET].values, test_pred, output_dir, tag)
    atomic_write_csv(importance_df, output_dir / f"{tag}_feature_importance.csv", index=False)
    model.booster_.save_model(str(output_dir / f"{tag}_model.txt"))

    return completed_metrics(
        {
            "cv_mean_r2": float(cv_df["R2"].mean()),
            "cv_std_r2": float(cv_df["R2"].std()),
            **test_metrics,
        }
    )


class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden=(256, 128, 64), dropout=0.15):
        super().__init__()
        layers = []
        previous = input_dim
        for hidden_size in hidden:
            layers.extend([nn.Linear(previous, hidden_size), nn.ReLU(), nn.Dropout(dropout)])
            previous = hidden_size
        layers.append(nn.Linear(previous, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(train_df, test_df, features, output_dir: Path, tag: str):
    output_dir.mkdir(parents=True, exist_ok=True)

    scaler = StandardScaler()
    x_train = scaler.fit_transform(train_df[features])
    x_test = scaler.transform(test_df[features])

    y_train = train_df[TARGET].values.astype(np.float32)
    y_test = test_df[TARGET].values.astype(np.float32)

    x_train_t = torch.tensor(x_train, dtype=torch.float32, device=DEVICE)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32, device=DEVICE)

    val_size = max(1, int(0.1 * len(x_train_t)))
    x_val_t = x_train_t[-val_size:]
    y_val_t = y_train_t[-val_size:]
    x_train_fold_t = x_train_t[:-val_size]
    y_train_fold_t = y_train_t[:-val_size]

    train_loader = DataLoader(
        TensorDataset(x_train_fold_t, y_train_fold_t),
        batch_size=4096 if HARDWARE["gpu_count"] >= 1 else 1024,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(x_val_t, y_val_t),
        batch_size=4096 if HARDWARE["gpu_count"] >= 1 else 1024,
        shuffle=False,
    )

    model = MLPRegressor(input_dim=x_train.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    best_val = float("inf")
    best_state = None
    patience = 15
    patience_counter = 0
    start_epoch = 0
    history = {"train_loss": [], "val_loss": []}
    checkpoint_path = mlp_checkpoint_file(output_dir, tag)

    if RESUME_ENABLED and checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        if checkpoint.get("input_dim") == x_train.shape[1] and checkpoint.get("features") == features:
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            history = checkpoint.get("history", history)
            best_val = checkpoint.get("best_val", best_val)
            best_state = checkpoint.get("best_state_dict")
            patience_counter = checkpoint.get("patience_counter", patience_counter)
            start_epoch = checkpoint.get("epoch", -1) + 1
            print(f"Resuming MLP from epoch {start_epoch}")

    for epoch in range(start_epoch, MAX_MLP_EPOCHS):
        model.train()
        train_loss = 0.0
        train_batches = 0

        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_batches += 1

        train_loss /= max(1, train_batches)

        model.eval()
        val_loss = 0.0
        val_batches = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                pred = model(xb)
                loss = criterion(pred, yb)
                val_loss += loss.item()
                val_batches += 1
        val_loss /= max(1, val_batches)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if ((epoch + 1) % MLP_CHECKPOINT_EVERY == 0) or (val_loss <= best_val):
            atomic_torch_save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_state_dict": best_state,
                    "best_val": best_val,
                    "patience_counter": patience_counter,
                    "history": history,
                    "input_dim": x_train.shape[1],
                    "features": features,
                },
                checkpoint_path,
            )

        if patience_counter >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        test_pred = model(x_test_t).cpu().numpy()

    test_metrics = evaluate_regression(y_test, test_pred)

    plt.figure(figsize=(8, 5))
    plt.plot(history["train_loss"], label="Train")
    plt.plot(history["val_loss"], label="Val")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title(f"MLP {tag} Loss Curve")
    plt.legend()
    plt.tight_layout()
    atomic_save_current_figure(output_dir / f"{tag}_loss_curve.png")
    plt.close()

    plot_predictions(y_test, test_pred, output_dir / f"{tag}_actual_vs_pred.png", f"MLP {tag} Actual vs Predicted")
    plot_residuals(y_test, test_pred, output_dir / f"{tag}_residuals.png", f"MLP {tag} Residuals")
    save_predictions(y_test, test_pred, output_dir, tag)

    atomic_torch_save(
        {
            "model_state_dict": model.state_dict(),
            "input_dim": x_train.shape[1],
            "features": features,
        },
        output_dir / f"{tag}_model.pt",
    )
    atomic_joblib_dump(scaler, output_dir / f"{tag}_scaler.joblib")

    if checkpoint_path.exists():
        checkpoint_path.unlink()

    return completed_metrics(test_metrics)


def build_summary_row(setup_name: str, model_name: str, metrics: dict[str, object]) -> dict[str, object]:
    row = {"setup": setup_name, "model": model_name}
    row.update(metrics)
    return row


def load_summary_map() -> dict[tuple[str, str], dict[str, object]]:
    summary_map: dict[tuple[str, str], dict[str, object]] = {}
    if not PARTIAL_RESULTS_PATH.exists():
        return summary_map

    frame = pd.read_csv(PARTIAL_RESULTS_PATH)
    for _, row in frame.iterrows():
        row_dict = {key: (None if pd.isna(value) else value) for key, value in row.to_dict().items()}
        key = (str(row_dict["setup"]), str(row_dict["model"]))
        summary_map[key] = row_dict
    return summary_map


def persist_summary_map(summary_map: dict[tuple[str, str], dict[str, object]], *, final: bool = False) -> None:
    if not summary_map:
        return

    frame = pd.DataFrame(summary_map.values()).sort_values(["setup", "model"]).reset_index(drop=True)
    atomic_write_csv(frame, PARTIAL_RESULTS_PATH, index=False)
    if final:
        atomic_write_csv(frame, FINAL_RESULTS_PATH, index=False)


def run_model(
    setup_name: str,
    model_name: str,
    trainer_fn,
    output_dir: Path,
    tag: str,
    state: dict[str, object],
    summary_map: dict[tuple[str, str], dict[str, object]],
) -> dict[str, object]:
    saved = load_saved_metrics(output_dir, tag)
    if saved is not None:
        print(f"Resuming from saved result for {setup_name}/{model_name}")
        summary_map[(setup_name, model_name)] = build_summary_row(setup_name, model_name, saved)
        persist_summary_map(summary_map)
        update_model_state(state, setup_name, model_name, output_dir, saved)
        return saved

    set_current_model(state, setup_name, model_name)

    try:
        metrics = trainer_fn()
        if not isinstance(metrics, dict):
            metrics = failed_metrics(f"Trainer for {model_name} returned a non-dict payload")
    except Exception as exc:
        metrics = failed_metrics(clean_reason(exc))
        print(f"{model_name} failed: {metrics['reason']}")

    save_metrics_bundle(output_dir, tag, metrics)
    update_model_state(state, setup_name, model_name, output_dir, metrics)
    summary_map[(setup_name, model_name)] = build_summary_row(setup_name, model_name, metrics)
    persist_summary_map(summary_map)
    return metrics


def main() -> None:
    start = time.time()

    print_sep()
    print("TRAINING TRANSACTION BEHAVIOR MODELS WITH RESUME SUPPORT")
    print(f"Input: {INPUT_PATH}")
    print(f"Target: {TARGET}")
    print(f"Device: {DEVICE}")
    print(f"GPU count: {HARDWARE['gpu_count']}")
    print(f"GPU names: {HARDWARE['gpu_names']}")
    print(f"Resume enabled: {RESUME_ENABLED}")

    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Input dataset not found: {INPUT_PATH}")

    df = pd.read_csv(INPUT_PATH, low_memory=False)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.sort_values(TIME_COL).reset_index(drop=True)

    df, clean_features, leaky_features = add_behavior_features(df)

    run_metadata = {
        "hardware": HARDWARE,
        "target": TARGET,
        "note": "This pipeline is valid for transaction behavior and gas usage modeling, not Health Factor forecasting.",
        "n_rows": int(len(df)),
        "n_wallets": int(df[GROUP_COL].nunique()),
        "resume_enabled": RESUME_ENABLED,
        "input_path": str(INPUT_PATH),
    }
    save_json(run_metadata, OUTPUT_ROOT / "run_metadata.json")

    experiments = {
        "clean": clean_features,
        "leaky": leaky_features,
    }

    state = load_run_state()
    summary_map = load_summary_map()

    for setup_name, feature_cols in experiments.items():
        print_sep()
        print(f"EXPERIMENT SETUP: {setup_name.upper()}")

        train_df, test_df, final_features = prepare_dataset(df, feature_cols)
        setup_dir = OUTPUT_ROOT / setup_name
        setup_dir.mkdir(parents=True, exist_ok=True)

        save_json(
            {
                "setup": setup_name,
                "n_train": int(len(train_df)),
                "n_test": int(len(test_df)),
                "n_features": len(final_features),
                "features": final_features,
            },
            setup_dir / "dataset_info.json",
        )

        models = {}

        models["RandomForest"] = run_model(
            setup_name,
            "RandomForest",
            lambda: train_random_forest(train_df, test_df, final_features, setup_dir / "RandomForest", setup_name),
            setup_dir / "RandomForest",
            setup_name,
            state,
            summary_map,
        )

        models["AdaBoost"] = run_model(
            setup_name,
            "AdaBoost",
            lambda: train_adaboost(train_df, test_df, final_features, setup_dir / "AdaBoost", setup_name),
            setup_dir / "AdaBoost",
            setup_name,
            state,
            summary_map,
        )

        models["XGBoost"] = run_model(
            setup_name,
            "XGBoost",
            lambda: train_xgboost(train_df, test_df, final_features, setup_dir / "XGBoost", setup_name),
            setup_dir / "XGBoost",
            setup_name,
            state,
            summary_map,
        )

        models["LightGBM"] = run_model(
            setup_name,
            "LightGBM",
            lambda: train_lightgbm(train_df, test_df, final_features, setup_dir / "LightGBM", setup_name),
            setup_dir / "LightGBM",
            setup_name,
            state,
            summary_map,
        )

        models["MLP"] = run_model(
            setup_name,
            "MLP",
            lambda: train_mlp(train_df, test_df, final_features, setup_dir / "MLP", setup_name),
            setup_dir / "MLP",
            setup_name,
            state,
            summary_map,
        )

        save_json(models, setup_dir / "model_summary.json")

    persist_summary_map(summary_map, final=True)

    summary_df = pd.DataFrame(summary_map.values())
    if not summary_df.empty:
        summary_df = summary_df.sort_values(["setup", "R2"], ascending=[True, False], na_position="last")
        print_sep()
        print("FINAL SUMMARY")
        print(summary_df.to_string(index=False))

    state["completed_at"] = time.time()
    save_run_state(state)

    print_sep()
    print(f"Total execution time: {time.time() - start:.2f} seconds")
    print(f"Saved outputs in: {OUTPUT_ROOT.resolve()}")


if __name__ == "__main__":
    main()

Writing 03_train_transaction_behavior_models.py


In [ ]:
!python -u 03_train_transaction_behavior_models.py



TRAINING TRANSACTION BEHAVIOR MODELS WITH RESUME SUPPORT
Input: data/cleaned/aave_combined_cleaned.csv
Target: GasUsed
Device: cpu
GPU count: 0
GPU names: []
Resume enabled: True


EXPERIMENT SETUP: CLEAN


In [ ]:
%%writefile 04_summarize_model_results.py
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


BASE_DIR = Path(".")
MODEL_OUTPUT_DIR = BASE_DIR / "model_outputs"
SUMMARY_DIR = BASE_DIR / "summary_outputs"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CANDIDATES = [
    MODEL_OUTPUT_DIR / "all_model_results.csv",
    MODEL_OUTPUT_DIR / "all_model_results.partial.csv",
]


def load_results() -> pd.DataFrame:
    for path in RESULTS_CANDIDATES:
        if path.exists():
            frame = pd.read_csv(path)
            if not frame.empty:
                print(f"Using results file: {path}")
                return frame
    raise FileNotFoundError("No result table found. Run the training cell first.")


def normalize_results(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame = frame.rename(columns={"R2": "r2", "RMSE": "rmse", "MAE": "mae", "MSE": "mse"})

    if "status" not in frame.columns:
        frame["status"] = "completed"
    if "reason" not in frame.columns:
        frame["reason"] = pd.NA

    for column in ["r2", "rmse", "mae", "mse", "cv_mean_r2", "cv_std_r2"]:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

    frame["setup"] = frame["setup"].astype(str)
    frame["model"] = frame["model"].astype(str)
    frame["status"] = frame["status"].fillna("completed").astype(str)

    display_names = {
        "RandomForest": "Random Forest",
        "AdaBoost": "AdaBoost",
        "XGBoost": "XGBoost",
        "LightGBM": "LightGBM",
        "MLP": "MLP",
    }
    frame["model_display"] = frame["model"].map(display_names).fillna(frame["model"])
    return frame


def completed_rows(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[frame["status"] == "completed"].copy()


def build_best_model_summary(frame: pd.DataFrame) -> pd.DataFrame:
    clean_rows = completed_rows(frame)
    clean_rows = clean_rows[clean_rows["setup"] == "clean"].copy()
    clean_rows = clean_rows.dropna(subset=["r2"])
    if clean_rows.empty:
        return pd.DataFrame()
    best_index = clean_rows["r2"].idxmax()
    return clean_rows.loc[[best_index]].reset_index(drop=True)


def build_leakage_comparison(frame: pd.DataFrame) -> pd.DataFrame:
    rows = completed_rows(frame)
    clean_rows = rows[rows["setup"] == "clean"][["model", "model_display", "r2", "rmse", "mae"]].copy()
    leaky_rows = rows[rows["setup"] == "leaky"][["model", "r2", "rmse", "mae"]].copy()

    merged = clean_rows.merge(leaky_rows, on="model", suffixes=("_clean", "_leaky"), how="inner")
    if merged.empty:
        return pd.DataFrame()

    merged["r2_inflation"] = merged["r2_leaky"] - merged["r2_clean"]
    merged["rmse_delta"] = merged["rmse_leaky"] - merged["rmse_clean"]
    merged["mae_delta"] = merged["mae_leaky"] - merged["mae_clean"]
    return merged.sort_values("model_display").reset_index(drop=True)


def write_latex_table(frame: pd.DataFrame, output_path: Path) -> None:
    clean_rows = completed_rows(frame)
    clean_rows = clean_rows[clean_rows["setup"] == "clean"].dropna(subset=["r2", "rmse"])
    if clean_rows.empty:
        print("No clean completed rows available for LaTeX export")
        return

    clean_rows = clean_rows.sort_values("r2", ascending=False)
    lines = [
        "\\begin{tabular}{lcc}",
        "\\toprule",
        "Model & $R^2$ & RMSE \\",
        "\\midrule",
    ]
    for _, row in clean_rows.iterrows():
        lines.append(f"{row['model_display']} & {row['r2']:.4f} & {row['rmse']:.4f} \\")
    lines.extend(["\\bottomrule", "\\end{tabular}"])
    output_path.write_text("\n".join(lines), encoding="utf-8")


def plot_clean_model_comparison(frame: pd.DataFrame, output_path: Path) -> None:
    clean_rows = completed_rows(frame)
    clean_rows = clean_rows[clean_rows["setup"] == "clean"].dropna(subset=["r2"])
    if clean_rows.empty:
        return

    clean_rows = clean_rows.sort_values("r2", ascending=False)
    plt.figure(figsize=(10, 6))
    plt.bar(clean_rows["model_display"], clean_rows["r2"])
    plt.ylabel("R^2")
    plt.xlabel("Model")
    plt.title("Clean Pipeline: Model Comparison")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_leakage_inflation(frame: pd.DataFrame, output_path: Path) -> None:
    if frame.empty:
        return

    plot_df = frame.sort_values("r2_inflation", ascending=False)
    plt.figure(figsize=(10, 6))
    plt.bar(plot_df["model_display"], plot_df["r2_inflation"])
    plt.ylabel("R^2 inflation")
    plt.xlabel("Model")
    plt.title("Leakage Impact on R^2")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_clean_vs_leaky(frame: pd.DataFrame, output_path: Path) -> None:
    rows = completed_rows(frame)
    rows = rows[rows["setup"].isin(["clean", "leaky"])].dropna(subset=["r2"])
    if rows.empty:
        return

    pivot = rows.pivot_table(index="model_display", columns="setup", values="r2", aggfunc="first")
    if pivot.empty:
        return

    x = np.arange(len(pivot.index))
    width = 0.35
    plt.figure(figsize=(10, 6))
    if "clean" in pivot.columns:
        plt.bar(x - width / 2, pivot["clean"], width, label="Clean")
    if "leaky" in pivot.columns:
        plt.bar(x + width / 2, pivot["leaky"], width, label="Leaky")
    plt.xticks(x, pivot.index, rotation=20, ha="right")
    plt.ylabel("R^2")
    plt.xlabel("Model")
    plt.title("Clean vs Leaky Performance")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()


def main() -> None:
    print("=" * 80)
    print("04_summarize_model_results.py")
    print("=" * 80)

    results = normalize_results(load_results())
    best_model = build_best_model_summary(results)
    leakage = build_leakage_comparison(results)

    summary_csv = SUMMARY_DIR / "all_model_results_summary.csv"
    best_csv = SUMMARY_DIR / "best_clean_model.csv"
    leakage_csv = SUMMARY_DIR / "leakage_comparison_summary.csv"

    results.to_csv(summary_csv, index=False)
    print(f"Wrote summary CSV: {summary_csv}")

    if not best_model.empty:
        best_model.to_csv(best_csv, index=False)
        print(f"Wrote best model CSV: {best_csv}")

    if not leakage.empty:
        leakage.to_csv(leakage_csv, index=False)
        print(f"Wrote leakage comparison CSV: {leakage_csv}")

    write_latex_table(results, SUMMARY_DIR / "paper_table_clean_models.tex")
    plot_clean_model_comparison(results, SUMMARY_DIR / "clean_model_r2_comparison.png")
    plot_leakage_inflation(leakage, SUMMARY_DIR / "leakage_r2_inflation.png")
    plot_clean_vs_leaky(results, SUMMARY_DIR / "clean_vs_leaky_r2.png")

    display_cols = [column for column in ["model_display", "setup", "status", "r2", "rmse", "mae", "cv_mean_r2"] if column in results.columns]
    print("\nAll results:")
    print(results[display_cols].sort_values(["setup", "r2"], ascending=[True, False], na_position="last").to_string(index=False))

    if not best_model.empty:
        print("\nBest clean model:")
        print(best_model[[column for column in display_cols if column in best_model.columns]].to_string(index=False))

    if not leakage.empty:
        print("\nLeakage comparison:")
        print(
            leakage[[
                column
                for column in ["model_display", "r2_clean", "r2_leaky", "r2_inflation", "rmse_clean", "rmse_leaky", "rmse_delta"]
                if column in leakage.columns
            ]].to_string(index=False)
        )

    print(f"\nOutputs saved in: {SUMMARY_DIR.resolve()}")


if __name__ == "__main__":
    main()

In [ ]:
!python -u 04_summarize_model_results.py

In [ ]:
import pandas as pd

df = pd.read_csv("data/cleaned/aave_combined_cleaned.csv")

stats = {
    "Total transactions": len(df),
    "Unique wallets (From)": df["From"].nunique() if "From" in df.columns else None,
    "V2 transactions": (df["Dataset"] == "V2").sum() if "Dataset" in df.columns else None,
    "V3 transactions": (df["Dataset"] == "V3").sum() if "Dataset" in df.columns else None,
    "Mean GasUsed": df["GasUsed"].mean() if "GasUsed" in df.columns else None,
    "Median GasUsed": df["GasUsed"].median() if "GasUsed" in df.columns else None,
    "Std GasUsed": df["GasUsed"].std() if "GasUsed" in df.columns else None,
    "Mean gas": df["gas"].mean() if "gas" in df.columns else None,
    "Median gas": df["gas"].median() if "gas" in df.columns else None,
}

summary_df = pd.DataFrame(stats.items(), columns=["Metric", "Value"])
summary_df

In [ ]:
import importlib.util
import sys
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

module_name = "train_module"
file_path = "03_train_transaction_behavior_models.py"
spec = importlib.util.spec_from_file_location(module_name, file_path)
train_module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = train_module
spec.loader.exec_module(train_module)

def evaluate_baseline(model_name, model, train_df, test_df, features):
    cv_r2 = []
    
    # 5-Fold Grouped CV
    for fold, train_idx, valid_idx in train_module.grouped_cv_splits(train_df, train_module.N_FOLDS):
        train_fold = train_df.iloc[train_idx]
        valid_fold = train_df.iloc[valid_idx]
        
        model.fit(train_fold[features], train_fold[train_module.TARGET])
        pred = model.predict(valid_fold[features])
        metrics = train_module.evaluate_regression(valid_fold[train_module.TARGET], pred)
        cv_r2.append(metrics['R2'])
        
    cv_mean = np.mean(cv_r2)
    cv_std = np.std(cv_r2)
    
    # Final Test Set Evaluation
    model.fit(train_df[features], train_df[train_module.TARGET])
    test_pred = model.predict(test_df[features])
    test_metrics = train_module.evaluate_regression(test_df[train_module.TARGET], test_pred)
    
    print(f"{model_name:<20} | {cv_mean:.3f}      | {cv_std:.3f}      | {test_metrics['R2']:.3f}    | {test_metrics['RMSE']:,.0f}")

# 1. Load and prep data exactly as the main pipeline
print("Loading and preparing dataset (this might take a moment)...")
df = pd.read_csv(train_module.INPUT_PATH, low_memory=False)
df[train_module.TIME_COL] = pd.to_datetime(df[train_module.TIME_COL], errors="coerce", utc=True)
df = df.sort_values(train_module.TIME_COL).reset_index(drop=True)

df, clean_features, _ = train_module.add_behavior_features(df)
train_df, test_df, final_features = train_module.prepare_dataset(df, clean_features)

print("\n" + "="*80)
print(f"{'Model':<20} | CV Mean R2 | CV Std Dev | Test R2   | Test RMSE")
print("-" * 80)

# 2. Linear Regression (Needs scaling for reliable convergence)
lr_model = make_pipeline(StandardScaler(), LinearRegression())
evaluate_baseline("Linear Regression", lr_model, train_df, test_df, final_features)

# 3. Decision Tree (Unpruned standard baseline)
dt_model = DecisionTreeRegressor(random_state=train_module.RANDOM_STATE)
evaluate_baseline("Decision Tree", dt_model, train_df, test_df, final_features)
print("="*80 + "\n")

In [ ]:
import importlib.util
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupShuffleSplit
import shap

# Dynamically import your training script
module_name = "train_module"
file_path = "03_train_transaction_behavior_models.py"
spec = importlib.util.spec_from_file_location(module_name, file_path)
train_module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = train_module
spec.loader.exec_module(train_module)

print("Loading and preparing dataset...")
df = pd.read_csv(train_module.INPUT_PATH, low_memory=False)
df[train_module.TIME_COL] = pd.to_datetime(df[train_module.TIME_COL], errors="coerce", utc=True)
df = df.sort_values(train_module.TIME_COL).reset_index(drop=True)
df, clean_features, _ = train_module.add_behavior_features(df)
train_df, test_df, final_features = train_module.prepare_dataset(df, clean_features)

def train_eval_lgbm(features_to_use, train_x, train_y, test_x, test_y):
    model = LGBMRegressor(n_estimators=300, random_state=42, n_jobs=-1, verbosity=-1)
    model.fit(train_x[features_to_use], train_y)
    pred = model.predict(test_x[features_to_use])
    return train_module.evaluate_regression(test_y, pred), model

print("\n--- 1. ABLATION STUDY ---")
feature_sets = {
    "Full Model": final_features,
    "No Wallet History": [f for f in final_features if f not in ["wallet_tx_count_so_far", "function_count_so_far", "Nonce"]],
    "No Temporal": [f for f in final_features if f not in ["Hour", "Day", "Month", "Year", "Confirmations"]],
    "No Gas Features": [f for f in final_features if f not in ["gas", "log_gas", "GasPrice", "log_GasPrice"]]
}

for name, feats in feature_sets.items():
    metrics, _ = train_eval_lgbm(feats, train_df, train_df[train_module.TARGET], test_df, test_df[train_module.TARGET])
    print(f"{name:<20} | R2: {metrics['R2']:.3f} | RMSE: {metrics['RMSE']:,.0f}")

print("\n--- 2. WALLET-LEVEL ROBUSTNESS TEST ---")
# Split 80% of wallets to train, 20% to test
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['Wallet']))
robust_train = df.iloc[train_idx].copy()
robust_test = df.iloc[test_idx].copy()

# Prepare sets avoiding overlap
rob_train_df, rob_test_df, rob_features = train_module.prepare_dataset(
    pd.concat([robust_train, robust_test]), clean_features
)
rob_train_df = rob_train_df[rob_train_df['Wallet'].isin(robust_train['Wallet'])]
rob_test_df = rob_test_df[rob_test_df['Wallet'].isin(robust_test['Wallet'])]

metrics, _ = train_eval_lgbm(final_features, rob_train_df, rob_train_df[train_module.TARGET], rob_test_df, rob_test_df[train_module.TARGET])
print(f"Unseen Wallets Split | R2: {metrics['R2']:.3f} | RMSE: {metrics['RMSE']:,.0f}")

print("\n--- 3. GENERATING FIGURES ---")
# Histogram
plt.figure(figsize=(8, 5))
plt.hist(np.log1p(df[train_module.TARGET].dropna()), bins=100, color='skyblue', edgecolor='black', alpha=0.7)
plt.title("Distribution of log(GasUsed)")
plt.xlabel("log(GasUsed + 1)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("gas_histogram.png", dpi=300)
print("Saved gas_histogram.png")

# SHAP (Using a 50k subsample to save time)
print("Calculating SHAP values (subsampled for speed)...")
sample_x = train_df[final_features].sample(min(50000, len(train_df)), random_state=42)
_, lgbm_model = train_eval_lgbm(final_features, train_df, train_df[train_module.TARGET], test_df, test_df[train_module.TARGET])
explainer = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(sample_x)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_x, show=False)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=300, bbox_inches='tight')
print("Saved shap_summary.png")